# Response Time Analysis (RTA)

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.impute import KNNImputer
import scipy.stats as stats
from scipy.stats import levene
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.api as sm
from statsmodels.formula.api import ols
import pingouin as pg

## 1. Importing datasets

In [ ]:
# setting parameters for Papermill
input_data_path = 'Results/preprocessed_data_with_distances.csv'
url2 = 'https://raw.githubusercontent.com/JeroenGuillierme/Project-MDA/main/Data/'
output_rta_path = 'Results/rta_data.csv'

In [ ]:
rta_df = pd.read_csv(input_data_path)

# Load Belgium with regions shapefile
belgium_with_provinces_boundary = gpd.read_file(f'{url2}BELGIUM_-_Provinces.geojson')

# Setting the style for the plots
sns.set(style="whitegrid")

## 2. Outlier Detection for Response Time T3-T0

Unsupervised learning method, Isolation forest, used. In this case only important how the model works on the given dataset, and not on new data. So, data splitting is not performed.

In [ ]:
interventions_data = rta_df[rta_df['Intervention'] == 1]
print(interventions_data['T3-T0'].isna().sum())

### 2.1 Visualisation distribution Response Times

In [ ]:
# Plot response times
sns.histplot(data=interventions_data['T3-T0'], bins=50, log_scale=True, kde=True).set(title='Logscale of Response Times', xlabel='Log(T3-T0)') 
# Right-skewed distribution, so log scale was used.

### 2.2 Running Isolation Forest

In [ ]:
# Split the DataFrame into two: one with the NaN and one without in the 'T3-T0' column
# DataFrame with NaN values
interventions_data_with_nan = interventions_data[interventions_data['T3-T0'].isna()]  
# DataFrame without NaN values
interventions_data_without_nan = interventions_data[~interventions_data['T3-T0'].isna()]  

print('Without NaN: ', len(interventions_data_without_nan))
print('With NaN: ', len(interventions_data_with_nan))

In [ ]:
Time = interventions_data_without_nan['T3-T0']

# IsolationForest algorithm
IsoFo = IsolationForest(n_estimators=100, contamination='auto',
                        random_state=45)  # Random state added for reproducibility
y_labels = IsoFo.fit_predict(np.array(Time).reshape(-1, 1))

# Only including the inliers
interventions_data_filtered = interventions_data_without_nan[y_labels == 1]  # DataFrame with inliers
discarded_rows = interventions_data_without_nan[y_labels == -1]  # DataFrame with outliers

min_timedelta = discarded_rows['T3-T0'].min()  
max_timedelta = discarded_rows['T3-T0'].max()
min_timedelta2 = interventions_data_filtered['T3-T0'].min()
max_timedelta2 = interventions_data_filtered['T3-T0'].max()
print(f"min response time of outliers: {min_timedelta}") 
print(f"max response time of outliers: {max_timedelta}")
print(f"min response time of inliers: {min_timedelta2}") 
print(f"max response time of inliers: {max_timedelta2}")
print(f"Number of discarded rows: {len(discarded_rows)}")
print(f"Number of filtered rows: {len(interventions_data_filtered)}")


In [ ]:
print("\nFiltered DataFrame (Inliers):")
interventions_data_filtered.head(5)

### 2.3 Total dataset without Response Time outliers

In [ ]:
# Add rows with NaN values again:
rta_ready = pd.concat([interventions_data_filtered, interventions_data_with_nan], axis=0)

# Reset index
rta_ready.reset_index(drop=True, inplace=True)

In [ ]:
# Plot distribution response times
sns.histplot(data=rta_ready['T3-T0'], bins=50, log_scale=True, kde=True).set(title='Logscale of Response Times', xlabel='Log(T3-T0)') 
# Right-skewed distribution, so log scale was used.

In [ ]:
# Last check if no anomalies are left in the dataset

# Get the minimum and maximum values of the 'latitude' column
min_latitude = rta_ready['Latitude'].min()
max_latitude = rta_ready['Latitude'].max()
# Get the minimum and maximum values of the 'longitude' column
min_longitude = rta_ready['Longitude'].min()
max_longitude = rta_ready['Longitude'].max()
# Get the minimum and maximum values of the response time column
min_timedelta = rta_ready['T3-T0'].min()
max_timedelta = rta_ready['T3-T0'].max()

print('Length of dataset: ', len(rta_ready))

print('Number of outliers/anomalies detected: ', len(discarded_rows))
print('Number of inliers: ', len(rta_ready))

print(f"Minimum latitude of dataset: {min_latitude}")
print(f"Maximum latitude of dataset: {max_latitude}")
print(f"Minimum longitude of dataset: {min_longitude}")
print(f"Maximum longitude of dataset: {max_longitude}")
print(f"min_timedelta of dataset: {min_timedelta}")
print(f"max_timedelta of dataset: {max_timedelta}")  

The Isolation Forest algorithm indicated every intervention with a response time longer than 43.5 minutes as an outlier/anomaly.

### 2.4 Visualisation coordinates of intervention locations with outlying Response Time

Most of the interventions with outlying Response Times are found in West-Vlaanderen and Vlaams Brabant.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=discarded_rows, x= 'Longitude', y='Latitude', hue='Province', ax=ax)
belgium_with_provinces_boundary.plot(ax=ax, facecolor='none', edgecolor='black')

### 2.5 Visualisation coordinates of intervention locations with inlying Response Times

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=interventions_data_filtered, x= 'Longitude', y='Latitude', hue='Province', ax=ax)
belgium_with_provinces_boundary.plot(ax=ax, facecolor='none', edgecolor='black')

## 3. Imputing missing Response Times

### 3.1 Visualisation coordinates of intervention locations with no reported Response Times

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=interventions_data_with_nan, x= 'Longitude', y='Latitude', hue='Province', ax=ax)
belgium_with_provinces_boundary.plot(ax=ax, facecolor='none', edgecolor='black')

print('The dataset is missing ', len(interventions_data_with_nan), 'Response Times')

### 3.2 Replacing Missing Response Times with (k-NN) Imputed values Based On Location and Distances
Assign Response Times to intervention locations with no reported Response Time in the dataset using k-NN Imputer.

* Feature Selection: Selected Latitude, Longitude,T3-T0 and the calculated distances for imputation. 
* Scaling: No scaling was done whatsoever because the imputed values shouldn’t be affected by scaling, because sometimes extreme values should have an effect on the response time on a certain location. And outlying response times have been removed already.
* KNN Imputation: KNN Imputer applied to the data to fill in the missing values. Again no train and test set was created because it should only work on given dataset and not on new data. This method of imputation uses the k-Nearest Neighbors approach. By default, a euclidean distance metric that supports missing values, is used to find the nearest neighbors
* The imputation was split up over the three different kinds of vector types: Ambulance, MUG and PIT Because, the distance to the closest MUG or PIT location shouldn't influence the imputed response time of an ambulance.

In [ ]:
# Split data into the three groups
g1 = rta_ready[rta_ready['Vector type'] == 'Ambulance']
g2 = rta_ready[rta_ready['Vector type'] == 'MUG']
g3 = rta_ready[rta_ready['Vector type'] == 'PIT']
g4 = rta_ready[rta_ready['Vector type'].isna()]
print('Missing values:\n', g4.isna().sum())

# Reset indices of groups
g1.reset_index(drop=True, inplace=True)
g2.reset_index(drop=True, inplace=True)
g3.reset_index(drop=True, inplace=True)

Group 4 consists of 80 missing vector types and for each of these observations is the response time also missing.
So, these will be removed from the dataset

In [ ]:
# Select features for imputation
# Only include distance to nearest vector type of that group
features_for_imputation1 = g1[['Latitude', 'Longitude', 'distance_to_ambulance', 'T3-T0']]
features_for_imputation2 = g2[['Latitude', 'Longitude', 'distance_to_mug', 'T3-T0']]
features_for_imputation3 = g3[['Latitude', 'Longitude', 'distance_to_pit', 'T3-T0']]

print('Number of observations for the vector types Ambulance, MUG and PIT respectively: \n', 
      len(features_for_imputation1), '\n', len(features_for_imputation2), '\n', len(features_for_imputation3))

In [ ]:
# set seed for allowing multiple runs with same outcome
np.random.seed(42)

# Initialize the KNN Imputer
knn_imputer = KNNImputer(n_neighbors=5)

# Perform KNN Imputation for each vector type
imputed_values1 = knn_imputer.fit_transform(features_for_imputation1)
imputed_values2 = knn_imputer.fit_transform(features_for_imputation2)
imputed_values3 = knn_imputer.fit_transform(features_for_imputation3)

# Create a dataframes with the imputed values
imputed_data1 = pd.DataFrame(imputed_values1, columns=['Latitude', 'Longitude', 'distance_to_ambulance', 'T3-T0'])
imputed_data2 = pd.DataFrame(imputed_values2, columns=['Latitude', 'Longitude', 'distance_to_mug', 'T3-T0'])
imputed_data3 = pd.DataFrame(imputed_values3, columns=['Latitude', 'Longitude', 'distance_to_pit', 'T3-T0'])


### 3.3 Assigning imputed Response Times back to original dataframes and concatenate them back together

In [ ]:
g1.loc[:,'T3-T0'] = imputed_data1['T3-T0']
g2.loc[:,'T3-T0'] = imputed_data2['T3-T0']
g3.loc[:,'T3-T0'] = imputed_data3['T3-T0']

rta_ready = pd.concat([g1, g2, g3], axis=0, ignore_index=True) # Group 4 isn't concatenated, leaving these observations behind for further analysis
print('Missing values:\n', rta_ready.isna().sum())

### 3.4 Visualisation distribution imputed Response Times

In [ ]:
# Plot response times
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(data=rta_ready, x='T3-T0', bins=50, log_scale=False, kde=True, hue='Vector type',ax=axes[0]).set(
    title='Imputed Response Times', xlabel='T3-T0') 
sns.histplot(data=rta_ready, x='T3-T0', bins=50, log_scale=True, kde=True, hue='Vector type',ax=axes[1]).set(
    title='Logscale of Imputed Response Times', xlabel='Log10(T3-T0)') 
# Right-skewed distribution, so log scale was used.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=rta_ready.sort_values(by='T3-T0', ascending=True), x= 'Longitude', y='Latitude', hue='T3-T0', palette='YlOrRd', ax=ax)
belgium_with_provinces_boundary.plot(ax=ax, facecolor='none', edgecolor='black')

## 4. Split up Response Times in three categories: Short, Medium and Long 

According to the literature, the ambulance response time is divided into 3 categories:
short (<4 min), medium (4-8 min), and long (>8 min). 
If the accident is in a multi-store building, this time will include some additional time called vertical response time. 
Vertical response time includes time interval from arrival on-scene to arrival at the patient's side.
Although shorter ambulance response time can increase medical effectiveness and satisfaction, 
it may reduce productivity due to the resources required and increased costs.

(link: https://www.researchgate.net/publication/362523045_Assessment_of_Ambulance_Response_Time_A_Study_of_Tabriz_Emergency_Medical_Center_Tabriz_City_Iran/fulltext/62fbbc80e3c7de4c34605bf1/Assessment-of-Ambulance-Response-Time-A-Study-of-Tabriz-Emergency-Medical-Center-Tabriz-City-Iran.pdf?origin=scientificContributions ).

In [ ]:
# Create new column RT_category conatining the three categories
# Initialize the RT_category column with an empty string
rta_ready['RT_category'] = ''

# Apply the categorization
rta_ready.loc[rta_ready['T3-T0'] < 4, 'RT_category'] = 'Short (<4 min)'
rta_ready.loc[(rta_ready['T3-T0'] >= 4) & (rta_ready['T3-T0'] <= 8), 'RT_category'] = 'Medium (4-8 min)'
rta_ready.loc[rta_ready['T3-T0'] > 8, 'RT_category'] = 'Long (>8 min)'

In [ ]:
# Create a table counting the number of interventions per Response Time category for Ambulances
counts = rta_ready.loc[rta_ready['Vector type'] == 'Ambulance','RT_category'].value_counts
pd.merge(counts(normalize=False), counts(normalize=True), left_index=True, right_index=True)

## 5. Saving Imputed data without outliers to new dataset

In [ ]:
rta_ready.to_csv(output_rta_path, index=False)

## 6. Comparing Response Times between different groups

In [ ]:
# Renaming some variables for performing ANOVA
rta_anova = rta_ready
rta_anova['response_time'] = rta_anova['T3-T0']
rta_anova['vector_type'] = rta_anova['Vector type']

### 6.1 Response Time versus Vector type (Ambulance, MUG or PIT)

#### 6.1.1 Summary Statistics & Visualisation

In [ ]:
# Initialize dataset for comparison between vector type
rta_anova_vt = rta_anova[['response_time', 'vector_type']].dropna()
rta_anova_vt.reset_index(drop=True, inplace=True) # Reset index 

# Log-Transform the variable Response Time, because is right-skewed
rta_anova_vt['log_response_time'] = rta_anova_vt['response_time'].transform(np.log10)

print(len(rta_anova),len(rta_anova_vt))

In [ ]:
# Basics Statistics per Group
rta_anova_vt.groupby('vector_type')['response_time'].describe()

In [ ]:
sns.set(style='whitegrid')
sns.pointplot(x='vector_type', y = 'response_time', data = rta_anova_vt, hue='vector_type') # errorbars: 95% confidence interval

# Calculate means
means = rta_anova_vt.groupby('vector_type')['response_time'].mean()

# Annotate the means next to the points
for i, mean in enumerate(means):
    plt.text(i, mean, f'{mean:.2f}', ha='left', va='bottom')

plt.xlabel('Vector Type')
plt.ylabel('Response Time')

#### 6.1.2 Analysis of Variance

In [ ]:
# ANOVA
formula1 = 'log_response_time ~ vector_type'
model1 = ols(formula1, data=rta_anova_vt).fit()
anova_table1 = sm.stats.anova_lm(model1, typ=2)
anova_table1

In [ ]:
# Post-hoc test if ANOVA is significant
# Tukey's honestly significantly differneced (HSD) test
# H0: No significant difference between the means of two groups
if anova_table1['PR(>F)'].iloc[0] < 0.05:
    tukey1 = pairwise_tukeyhsd(rta_anova_vt['log_response_time'], rta_anova_vt['vector_type'], alpha=0.05)

tukey1.summary()

In [ ]:
# Plot group confidence intervals
tukey1.plot_simultaneous(comparison_name='MUG'); 

#### 6.1.3 Checking Model Assumptions

**Assumptions of ANOVA**
The ANOVA test has important assumptions that must be satisfied in order for the associated p-value to be valid.
* The samples are independent.
* Each sample is from a normally distributed population.
* The population standard deviations of the groups are all equal. This property is known as homoscedasticity.

**Normality Of Residuals**

Seems like normailty is violated.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Normality of residuals with QQ-plot
residuals1 = model1.resid
sm.qqplot(residuals1, line='45', ax=axes[0])
axes[0].set_title('Q-Q Plot of Residuals')
axes[0].set_xlabel("Theoretical Quantiles")
axes[0].set_ylabel("Standardized Residuals")

# Histogram
sns.histplot(residuals1, kde=True, ax=axes[1])

# Perform the Kolmogorov-Smirnov test for normality, because large dataset
ks_test1 = stats.kstest(residuals1, 'norm')

# Print the p-value
print('Kolmogorov-Smirnov Test: p-value =', ks_test1.pvalue)

# Interpret the result
if ks_test1.pvalue > 0.05:
    print("Residuals appear to be normally distributed (fail to reject H0).")
else:
    print("Residuals do not appear to be normally distributed (reject H0).")

**Homogeneity Of Variances**

In [ ]:
# Residuals vs Fitted Plot
fitted_vals1 = model1.fittedvalues
sns.residplot(x=fitted_vals1, y=residuals1, line_kws={'color': 'red'})
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('Residuals vs. Fitted Plot')


# Levene's test
# H0: population variances across groups are eaqual
levene_test1 = levene(rta_anova_vt['log_response_time'][rta_anova_vt['vector_type'] == 'Ambulance'],
                     rta_anova_vt['log_response_time'][rta_anova_vt['vector_type'] == 'MUG'],
                     rta_anova_vt['log_response_time'][rta_anova_vt['vector_type'] == 'PIT'])
print('p-value of the levene test: ', levene_test1[1])
# Interpret the result
if levene_test1.pvalue > 0.05:
    print("Variances across groups are equal (fail to reject H0).")
else:
    print("Variances across groups are not equal (reject H0).")

#### 6.1.4 Welch's ANOVA as alternative

Welch’s ANOVA compares two means to see if they are equal. It is an alternative to the Classic ANOVA and can be used even if your data violates the assumption of homogeneity of variances.

However, Welch's ANOVA assumes normality of the data within groups. Although, with large sample sizes, the Central Limit Theorem (CLT) plays a role and the normality assumption becomes less stringent.

**Welch's ANOVA if Homogeneity of Variances Rejected**

In [ ]:
# Apply Welch's ANOVA
welch1 = pg.welch_anova(dv='log_response_time', between='vector_type', data=rta_anova_vt)
welch1 # p-unc is the p-value; here: very small => reject the null hypothesis that the response time are equal between the three vector types.

**Post-Hoc Games Howell test if Welch's ANOVA significant**

In [ ]:
# If Welch's ANOVA significant:
# Games-Howell Post Hoc test for Welch's ANOVA
if welch1['p-unc'][0] < 0.05:
    gh1 = pg.pairwise_gameshowell(dv='log_response_time', between='vector_type', data=rta_anova_vt)
gh1

**Visualization of the p-values in the form of a Heatmap**

In [ ]:
# Visualize p-values in the form of a heatmap
# Pivot the results to get the p-values
pval_matrix1 = gh1.pivot(index='A', columns='B', values='pval').round(3)

# Plot the heatmap
sns.set(style='white')
plt.figure(figsize=(8, 6))
sns.heatmap(pval_matrix1, annot=True, cmap='coolwarm', cbar_kws={'label': 'p-value'})
plt.title('Pairwise p-values (Games-Howell)')
plt.show()

#### 6.1.5 Conclusion

After checking the model assumptions for a one-way ANOVA, we saw that the normality of the residuals and the homogeneity of the variances between the vector types weren't equal. So, the results of the ANOVA may be incorrect or misleading. 

Because the homogeneity was not met an alternative test was performed, namely the Welch's ANOVA. This test gave, just as the one-way ANOVA a significant result, indicating that the average response time for the different vector types aren't equal. As post-hoc test, a Games-Howell test was run to see where the differences are.

**Result**

Only the mean response time for the vector type MUG differs significantly from these of the Ambulances and PITs. The mean response times for MUGs are almost 15 minutes, which is (around 1.5 minutes) longer than the other vector types.

### 6.2 Only keep the one observation per Mission ID with the shortest response time

Only one observation for each Mission ID should be kept, namely the observation of the vector type with the shortest response time. Otherwise, multiple response times for the same mission will be used in the analysis, which can bias the results. When, for the same Mission ID, the different vector types have the same response time, the observation of the Ambulance will be prioritized. The observation with the shortest response time is chosen because it is the first emergency service present on the incident location that matters.

In [ ]:
# Sort by Response Time (T3-T0) first and then by vector type to prioritize Ambulance if the times are equal
# Remove duplicate Mission IDs, keeping only the observation with the shortest response time
rta_anova_sorted = rta_anova.sort_values(by=['response_time', 'vector_type'], ascending=[True, True]).drop_duplicates(
    subset='Mission ID', keep='first')

# Reset indices
rta_anova_sorted.reset_index(drop=True, inplace= True)

print('There are', len(rta_anova_sorted), 'unique Mission IDs')
rta_anova_sorted.head(5)

### 6.3 Response Time versus Province

#### 6.3.1 Summary Statistics & Visualisation

In [ ]:
# Discard rows with Missing Values for vector type
rta_anova_prov = rta_anova_sorted[['response_time', 'Province']].dropna()
rta_anova_prov.reset_index(drop=True, inplace=True) # Reset index after dropping missing values

# Log-Transform the variable Response Time, because is right-skewed
rta_anova_prov['log_response_time'] = np.log10(rta_anova_prov['response_time']) # Create new column for transformed response times

print(len(rta_anova_sorted),len(rta_anova_prov))

In [ ]:
# Basics Statistics per Group
rta_anova_prov.groupby('Province')['response_time'].describe()

In [ ]:
sns.set(style='whitegrid')
# Determine the order of provinces based on how they appear in the data
province_order = rta_anova_prov['Province'].unique()

# Create the point plot with a specified order
ax = sns.pointplot(x='Province', y='response_time', data=rta_anova_prov, hue='Province', order=province_order)

# Calculate means, ensuring the same order is used
means = rta_anova_prov.groupby('Province')['response_time'].mean().reindex(province_order)

# Annotate the means next to the points, in the correct order
for i, (province, mean) in enumerate(means.items()):
    plt.text(i, mean, f'{mean:.2f}', ha='left', va='bottom')

plt.ylabel('Response Time')
plt.xticks(rotation=90)

#### 6.3.2 Analysis of Variance

In [ ]:
# ANOVA
formula2 = 'log_response_time ~ Province'
model2 = ols(formula2, data=rta_anova_prov).fit()
anova_table2 = sm.stats.anova_lm(model2, typ=2)
anova_table2

In [ ]:
# Post-hoc test if ANOVA is significant
# Tukey's honestly significantly differneced (HSD) test
# H0: No significant difference between the means of two groups
if anova_table2['PR(>F)'].iloc[0] < 0.05:
    tukey2 = pairwise_tukeyhsd(rta_anova_prov['log_response_time'], rta_anova_prov['Province'], alpha=0.05)

tukey2.summary()

In [ ]:
# Plot group confidence intervals
tukey2.plot_simultaneous(comparison_name='Vlaams Brabant');  

#### 6.3.3 Checking Model Assumptions

**Normality Of Residuals**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Normality of residuals with QQ-plot
residuals2 = model2.resid
sm.qqplot(residuals2, line='45', ax=axes[0])
axes[0].set_title('Q-Q Plot of Residuals')
axes[0].set_xlabel("Theoretical Quantiles")
axes[0].set_ylabel("Standardized Residuals")

# Histogram
sns.histplot(residuals2, kde=True, ax=axes[1])

# Perform the Kolmogorov-Smirnov test for normality, because large dataset
ks_test2 = stats.kstest(residuals2, 'norm')

# Print the p-value
print('Kolmogorov-Smirnov Test: p-value =', ks_test2.pvalue)

# Interpret the result
if ks_test2.pvalue > 0.05:
    print("Residuals appear to be normally distributed (fail to reject H0).")
else:
    print("Residuals do not appear to be normally distributed (reject H0).")

**Homogeneity Of Variances**

In [ ]:
# Residuals vs Fitted Plot
fitted_vals2 = model2.fittedvalues
sns.residplot(x=fitted_vals2, y=residuals2, line_kws={'color': 'red'})
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('Residuals vs. Fitted Plot')


# Levene's test
# H0: population variances across groups are eaqual
levene_test2 = levene(rta_anova_prov['log_response_time'][rta_anova_prov['Province'] == 'Antwerpen'],
                     rta_anova_prov['log_response_time'][rta_anova_prov['Province'] == 'Limburg'],
                     rta_anova_prov['log_response_time'][rta_anova_prov['Province'] == 'Brabant Wallon'],
                     rta_anova_prov['log_response_time'][rta_anova_prov['Province'] == 'Vlaams Brabant'],
                     rta_anova_prov['log_response_time'][rta_anova_prov['Province'] == 'West-Vlaanderen'],
                     rta_anova_prov['log_response_time'][rta_anova_prov['Province'] == 'Hainaut'],
                     rta_anova_prov['log_response_time'][rta_anova_prov['Province'] == 'Namur'],
                     rta_anova_prov['log_response_time'][rta_anova_prov['Province'] == 'Liège'],
                     rta_anova_prov['log_response_time'][rta_anova_prov['Province'] == 'Luxembourg'],
                     rta_anova_prov['log_response_time'][rta_anova_prov['Province'] == 'Oost-Vlaanderen'],
                     rta_anova_prov['log_response_time'][rta_anova_prov['Province'] == 'Bruxelles'])
print('p-value of the levene test: ', levene_test2[1])
# Interpret the result
if levene_test2.pvalue > 0.05:
    print("Variances across groups are equal (fail to reject H0).")
else:
    print("Variances across groups are not equal (reject H0).")

#### 6.3.4 Welch's ANOVA as alternative

**Welch's ANOVA if Homogeneity of Variances Rejected**

In [ ]:
# Apply Welch's ANOVA
welch2 = pg.welch_anova(dv='log_response_time', between='Province', data=rta_anova_prov) 
welch2 # p-unc is the p-value; here: very small => reject the null hypothesis that the response time are equal between the eleven provinces.

In [ ]:
# If Welch's ANOVA significant:
# Games-Howell Post Hoc test for Welch's ANOVA
if welch2['p-unc'][0] < 0.05:
    gh2 = pg.pairwise_gameshowell(dv='log_response_time', between='Province', data=rta_anova_prov)
gh2

**Visualization of the p-values in the form of a Heatmap**

In [ ]:
# Pivot the results to get the p-values
pval_matrix2 = gh2.pivot(index='A', columns='B', values='pval').round(3)

# Plot the heatmap
sns.set(style='white')
plt.figure(figsize=(8, 6))
sns.heatmap(pval_matrix2, annot=True, cmap='coolwarm', cbar_kws={'label': 'p-value'})
plt.title('Pairwise p-values (Games-Howell)')
plt.show()

#### 6.3.5 Conclusion

After checking the model assumptions for a one-way ANOVA, we saw that the normality of the residuals and the homogeneity of the variances between the provinces weren't equal. So, the results of the ANOVA may be incorrect or misleading. 

Because the homogeneity was not met an alternative test was performed, namely the Welch's ANOVA. This test gave, just as the one-way ANOVA a significant result, indicating that the average response time for the different provinces aren't equal. As post-hoc test, a Games-Howell test was executed to see where the differences lie.

**Result**

Most remarkable observation is that the mean response time in Vlaams Brabant is significantly higher than all others. The province Vlaams Brabant has the longest response times, with an average of 15.7 minutes. Although, the provinces Oost- and West-Vlaanderen also stand out from the other provinces with longer response times, while their mean response time is approximately the same. Antwerp has the lowest response times, almost significantly smaller than all other provinces.

### 6.4 Response Time versus Event Level

#### 6.4.1 Summary Statistics & Visualisation

In [ ]:
# Discard rows with Missing Values for vector type
rta_anova_el = rta_anova_sorted[['response_time', 'Eventlevel']].dropna()

# Reset index after dropping missing values
rta_anova_el.reset_index(drop=True, inplace=True) 

# Make Event level categorical
rta_anova_el['Eventlevel'] = rta_anova_el['Eventlevel'].astype('category')

# Log-Transform the variable Response Time, because is right-skewed
rta_anova_el['log_response_time'] = np.nan # Initialize new column for transformed response times
rta_anova_el.loc[:,'log_response_time'] = np.log10(rta_anova_el['response_time'])

print(len(rta_anova_sorted),len(rta_anova_el))

In [ ]:
# Basics Statistics per Group
rta_anova_el.groupby('Eventlevel', observed=True)['response_time'].describe()

In [ ]:
sns.set(style='whitegrid')

# Determine the order of provinces based on how they appear in the data
eventlevel_order = rta_anova_el['Eventlevel'].unique().sort_values()

# Create the point plot with a specified order
ax = sns.pointplot(x='Eventlevel', y='response_time', data=rta_anova_el, hue='Eventlevel', order=eventlevel_order,
                  legend=False)

# Calculate means, ensuring the same order is used
means = rta_anova_el.groupby('Eventlevel',  observed=True)['response_time'].mean().reindex(eventlevel_order)

# Annotate the means next to the points, in the correct order
for i, (eventlevel, mean) in enumerate(means.items()):
    plt.text(i, mean, f'{mean:.2f}', ha='left', va='bottom')
    
plt.xlabel('Event Level')
plt.ylabel('Response Time')

#### 6.4.2 Analysis of Variance

In [ ]:
# ANOVA
formula3 = 'log_response_time ~ Eventlevel'
model3 = ols(formula3, data=rta_anova_el).fit()
anova_table3 = sm.stats.anova_lm(model3, typ=2)
anova_table3

In [ ]:
# Post-hoc test if ANOVA is significant
# Tukey's honestly significantly differneced (HSD) test
# H0: No significant difference between the means of two groups
if anova_table3['PR(>F)'].iloc[0] < 0.05:
    tukey3 = pairwise_tukeyhsd(rta_anova_el['log_response_time'], rta_anova_el['Eventlevel'], alpha=0.05)

tukey3.summary()

In [ ]:
# Plot group confidence intervals
tukey3.plot_simultaneous(comparison_name=0.0);    

#### 6.4.3 Checking Model Assumptions

**Normality Of Residuals**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Normality of residuals with QQ-plot
residuals3 = model3.resid
sm.qqplot(residuals3, line='45', ax=axes[0])
axes[0].set_title('Q-Q Plot of Residuals')
axes[0].set_xlabel("Theoretical Quantiles")
axes[0].set_ylabel("Standardized Residuals")

# Histogram
sns.histplot(residuals3, kde=True, ax=axes[1])

# Perform the Kolmogorov-Smirnov test for normality, because large dataset
ks_test3 = stats.kstest(residuals3, 'norm')

# Print the p-value
print('Kolmogorov-Smirnov Test: p-value =', ks_test3.pvalue)

# Interpret the result
if ks_test3.pvalue > 0.05:
    print("Residuals appear to be normally distributed (fail to reject H0).")
else:
    print("Residuals do not appear to be normally distributed (reject H0).")

**Homogeneity Of Variances**

In [ ]:
# Residuals vs Fitted Plot
fitted_vals3 = model3.fittedvalues
sns.residplot(x=fitted_vals3, y=residuals3, line_kws={'color': 'red'})
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('Residuals vs. Fitted Plot')


# Levene's test
# H0: population variances across groups are eaqual
levene_test3 = levene(rta_anova_el['log_response_time'][rta_anova_el['Eventlevel'] == 0.0],
                      rta_anova_el['log_response_time'][rta_anova_el['Eventlevel'] == 1.0],
                      rta_anova_el['log_response_time'][rta_anova_el['Eventlevel'] == 2.0],
                      rta_anova_el['log_response_time'][rta_anova_el['Eventlevel'] == 5.0],
                      rta_anova_el['log_response_time'][rta_anova_el['Eventlevel'] == 6.0],
                      rta_anova_el['log_response_time'][rta_anova_el['Eventlevel'] == 7.0])
print('p-value of the levene test: ', levene_test3[1])
# Interpret the result
if levene_test3.pvalue > 0.05:
    print("Variances across groups are equal (fail to reject H0).")
else:
    print("Variances across groups are not equal (reject H0).")

#### 6.4.4 Welch's ANOVA as alternative

**Welch's ANOVA if Homogeneity of Variances Rejected**

In [ ]:
# Apply Welch's ANOVA
welch3 = pg.welch_anova(dv='log_response_time', between='Eventlevel', data=rta_anova_el) 
welch3 # p-unc is the p-value; here: very small => reject the null hypothesis that the response time are equal between the eleven provinces.

In [ ]:
# If Welch's ANOVA significant:
# Games-Howell Post Hoc test for Welch's ANOVA
if welch3['p-unc'][0] < 0.05:
    gh3 = pg.pairwise_gameshowell(dv='log_response_time', between='Eventlevel', data=rta_anova_el)
gh3

**Visualization of the p-values in the form of a Heatmap**

In [ ]:
# Pivot the results to get the p-values
pval_matrix3 = gh3.pivot(index='A', columns='B', values='pval').round(3)

# Plot the heatmap
sns.set(style='white')
plt.figure(figsize=(8, 6))
sns.heatmap(pval_matrix3, annot=True, cmap='coolwarm', cbar_kws={'label': 'p-value'})
plt.title('Pairwise p-values (Games-Howell)')
plt.show()

#### 6.4.5 Conclusion

After checking the model assumptions for a one-way ANOVA, we saw that the normality of the residuals was rejected and the homogeneity of the variances between the provinces weren't equal. So, the results of the ANOVA may be incorrect or misleading. 

Because the homogeneity was not met an alternative test was performed, namely the Welch's ANOVA. This test gave, just as the one-way ANOVA a significant result, indicating that the average response time for the different event severeness levels aren't equal. As post-hoc test, a Games-Howell test was executed to see where the differences lie.

**Result**

Most remarkable observation is that the mean response times for the extreme event levels, zero and seven, differ significantly from the other levels. Here level seven has the longest mean response times and level zero the shortest. The average response time increases in general from event level zero to seven. This could indicate that a level zero cardiac arrest is worse than a level seven cardiac arrest. 